### Set up the remote kernel
If you use a remote kernel, then you need to set it up with all local code and
the corresponding required dependencies so that we can run our LLM in the Colab
while using the GPU.

In [ ]:
import sys
sys.path

#### Clone repositories

In [ ]:
!git clone https://github.com/NiccoloSacchi/assignment1-basics.git
print()
!git clone https://github.com/NiccoloSacchi/assignment2-systems.git
print()

In [ ]:
# Install Assignment 2's dependencies:
# --system: directly into the environment the Colab kernel is already using.
# -e: in "editable" mode, if you change code in the Assignment 1 folder, the
#   changes are reflected immediately without a reinstall.
!cd assignment2-systems; uv pip install --system -e .

!!!! **Finally, restart the kernel!** !!!!

#### Update repositories

In [ ]:
!cd assignment1-basics; git pull
print()
!cd assignment2-systems; git pull

### GPU

#### Summaries

In [ ]:
import torch
torch.cuda.get_device_properties(0)

In [ ]:
!nvidia-smi

In [ ]:
import torch
print(torch.cuda.memory_summary(device=None, abbreviated=False))

#### Memory release

In [ ]:
from cs336_systems.prettyprint import print_gpu_memory

print_gpu_memory()

# Create a tensor on the GPU.
c = 2048
r = 1024
tensor = torch.randn(r, c, dtype=torch.float32, device="cuda")
print(f"Initialized tensor of size {r*c*32/(8*1024**2)} MB.")
print_gpu_memory()

# Delete the tensor.
del tensor
print("Deleted tensor.")

# Allocated memory is released but reserved memory is not, it will be used for
# future allocations.
print_gpu_memory()

# Force PyTorch to release all unused reserved memory back to the GPU driver.
torch.cuda.empty_cache()

# Also reserved memory is released.
print_gpu_memory()

#### Arithmetics and backpropagation

In [ ]:
from cs336_systems.prettyprint import print_gpu_memory

# Check the GPU memory.
print_gpu_memory()

# Create 2 tensors and multiply.
c = 2048
r = 1024
x = torch.randn(r, c, dtype=torch.float32, requires_grad=True, device="cuda")
y = torch.randn(c, r, dtype=torch.float32, requires_grad=True, device="cuda")
print(f"Initialized 2 tensors of total size 2*{r*c*32/(8*1024**2)} MB.")
print_gpu_memory()

# Multiply them. There is an additional ~8MB. PyTorch initializes cuBLAS. This
# library loads kernels and allocates internal workspace buffers to handle the
# matrix multiplication efficiently.
z = x @ y
print(f"Multiplied the 2 tensors into a new tensor of size {r*r*32/(8*1024**2)} MB.")
print_gpu_memory()

# Sum itseld them.
k = z + z
print(f"Summed the tensor into a new tensor of size {r*r*32/(8*1024**2)} MB.")
print_gpu_memory()

# Backpropagate everything, thus populating grads.
k.sum().backward()
print("Backward propagation.")
print_gpu_memory()

# Delete all tensors and release memory.
del x, y, z, k
print("Deleted all tensors.")
torch.cuda.empty_cache()
print_gpu_memory()

#### Intermediary results
Intermediary results are stored when grad is enabled. Here we have only 1
variable of  size 4MB, but we consume 20MB.

In [ ]:
from cs336_systems.prettyprint import print_gpu_memory

# Check the GPU memory.
print_gpu_memory()

# Create 2 tensors and multiply.
c = 2048
r = 1024
x = (
  torch.randn(r, c, dtype=torch.float32, requires_grad=True, device="cuda") @
  torch.randn(c, r, dtype=torch.float32, requires_grad=True, device="cuda")
)
print(f"Multiplied in place 2 tensors of total size 2*{r*c*32/(8*1024**2)} MB into a new tensor of size {r*r*32/(8*1024**2)} MB.")
print_gpu_memory()

# Delete all tensors and release memory.
del x
print("Deleted all tensors.")
torch.cuda.empty_cache()
print_gpu_memory()

Intermediary results are *not* stored when requires_grad=False. Here we consume
exactly the size of the output tensor, 4MB.

In [ ]:
from cs336_systems.prettyprint import print_gpu_memory

# Check the GPU memory.
print_gpu_memory()

# Create 2 tensors and multiply.
c = 2048
r = 1024
x = (
  torch.randn(r, c, dtype=torch.float32, requires_grad=False, device="cuda") @
  torch.randn(c, r, dtype=torch.float32, requires_grad=False, device="cuda")
)
print(f"Multiplied in place 2 tensors of total size 2*{r*c*32/(8*1024**2)} MB into a new tensor of size {r*r*32/(8*1024**2)} MB.")
print_gpu_memory()

# Delete all tensors and release memory.
del x
print("Deleted all tensors.")
torch.cuda.empty_cache()
print_gpu_memory()

#### Hard clean

In [ ]:
# Copy-paste the output to Gemini and let it figure out what variable still need
# to be deleted.
locals()

In [ ]:
# Alteratively: restart the kernel.
import os

# Kills the current process and any other python processes on the VM to make
# sure no process is using the GPU anymore.
os.system("pkill -9 python")